# Titans memory on temporal data — does accumulation track predictability?

**What this tests.** `titans_real_text.py` found that a memory trained on a toy recall
task *forgets* (`norm_ratio` ~0.07) while one trained on enwik8 *accumulates*
(`norm_ratio` ~2.2). Two mechanisms were tested and ruled out, leaving:

> *whatever produces the accumulation is baked in by what the memory was trained on*

That was never tested directly, because there were only two corpora. This notebook
sweeps a **controlled knob on temporal structure** — AR(1) with increasing phi, plus
a periodic signal and a real forecasting benchmark — and asks whether `norm_ratio`
tracks how predictable the data is.

**The trick that makes it a fair test: quantile binning.** Every corpus is mapped to
256 equal-frequency bins, so all of them have near-identical *marginal* entropy
(~5.55 nats). Any difference in validation loss therefore comes from **temporal
dependence alone**, not from the shape of the value distribution. One variable moves.

**Architecture is unchanged** from `titans_real_text.py`: same `MemoryAsContextTransformer`,
dim 64, 64->256->64 memory MLP, `num_tokens=256`. The pipeline consumes a 1-D stream of
ints 0-255; we only change what produces that stream.

Runtime: ~10-20 min per corpus on a T4 at 2000 steps. Results checkpoint to JSON
after each corpus, so a disconnect doesn't cost you the run.

## 1 · Setup

In [ ]:
!pip install -q titans-pytorch
!git clone -q https://github.com/thebnbrkr/marv-titan.git /content/marv-titan
import sys; sys.path.insert(0, '/content/marv-titan/experiments')

import torch, numpy as np
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import gzip, json, time, urllib.request, os
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- the SAME code that produced the enwik8 baseline, imported, not copied ---
from titans_real_text import (
    build_model, sample_batch, train, _cos,
    load_enwik8 as _load_enwik8_from_path,
    diff_memory_on_passage, inspect_decay_gate, consecutive_write_alignment,
    DIM_HEAD, HIDDEN,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- matched to experiments/titans_real_text.py's own CLI defaults ---
CHECKPOINTS = [500, 1000, 2000, 3000]   # measure at each; last one is the headline
STEPS      = CHECKPOINTS[-1]            # the reported enwik8 runs used 3000
SEQ_LEN    = 256
BATCH      = 8
LR         = 2e-4
SEEDS      = [0]       # use [0, 1, 2] for a result; 1 seed for a first pass
N_BINS     = 256
PASSAGE_LEN = 1024

RESULTS_PATH = "temporal_sweep_results.json"
print("device:", DEVICE, "| memory:", DIM_HEAD, "->", HIDDEN, "->", DIM_HEAD)

## 2 · Quantile binning — the control that makes corpora comparable

`np.digitize` against 255 equal-frequency edges computed **on the training split only**
(no leakage from val). After this, every corpus is a stream of ints 0-255 with an
approximately uniform marginal — so a model's val loss reflects only how much the
*past predicts the future*.

In [ ]:
def quantize(series, n_bins=N_BINS, split=0.9):
    """Float series -> (train_tokens, val_tokens) as int64 in [0, n_bins).
    Bin edges come from the TRAIN portion only."""
    series = np.asarray(series, dtype=np.float64)
    series = series[np.isfinite(series)]
    k = int(len(series) * split)
    tr_raw, va_raw = series[:k], series[k:]
    edges = np.quantile(tr_raw, np.linspace(0, 1, n_bins + 1)[1:-1])
    tr = np.digitize(tr_raw, edges)
    va = np.digitize(va_raw, edges)
    return torch.from_numpy(tr).long(), torch.from_numpy(va).long()


def marginal_entropy_nats(tokens, n_bins=N_BINS):
    """Sanity check that binning really did equalise the marginal."""
    counts = np.bincount(tokens.numpy(), minlength=n_bins).astype(np.float64)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

## 3 · Corpora

Four families. **The AR(1) group is the actual experiment** -- only phi changes across
those four, so they alone support a causal claim. Everything else is supporting evidence.

| corpus | what it is | role |
|---|---|---|
| `ar_phi0.00` | i.i.d. Gaussian | controlled sweep (lower anchor) |
| `ar_phi0.50` / `0.90` / `0.99` | AR(1), `x_t = phi*x_{t-1} + e_t` | controlled sweep |
| `periodic` | sinusoid + noise | long-range vs short-range structure |
| `ETTm1` | real forecasting benchmark | does it hold on real data |
| `enwik8` | text | **reproduction anchor, NOT a controlled point** |

### Why enwik8 sits outside the control

Quantile binning assumes values are *ordered magnitudes* -- it sorts them and cuts at
equal frequencies. Byte values are **categories, not quantities**: `0x41` ('A') is not
"larger" than `0x20` (space), so sorting them is meaningless. enwik8 therefore enters as
raw bytes and keeps its own (lower) marginal entropy.

Its job here is to check that the pipeline still reproduces the previously observed
norm_ratio ~2.2. Do **not** read a text-vs-AR difference as evidence about temporal
structure -- the marginals differ, so that comparison is confounded. The AR sweep is
where the marginals are matched and the inference is clean.

In [ ]:
def ar1(n, phi, seed=0):
    rng = np.random.default_rng(seed)
    e = rng.standard_normal(n)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + e[t]
    return x


def periodic(n, period=97, noise=0.3, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    return np.sin(2 * np.pi * t / period) + 0.5 * np.sin(2 * np.pi * t / (period * 3.7)) \
           + noise * rng.standard_normal(n)


def _download(url, path):
    """urlretrieve, falling back to a certifi CA bundle when the system store is
    incomplete (common on local macOS Python; Colab works either way)."""
    if os.path.exists(path):
        return path
    try:
        urllib.request.urlretrieve(url, path)
    except Exception:
        import ssl, certifi, shutil
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(url, context=ctx) as r, open(path, "wb") as f:
            shutil.copyfileobj(r, f)
    return path


def lag1_mi(tok, coarse=16):
    """Lag-1 mutual information (nats) on coarsened bins -- a MODEL-FREE measure of
    how much the previous step tells you about the next. Pure property of the data."""
    x = (tok.numpy().astype(int) * coarse) // N_BINS
    J = np.histogram2d(x[:-1], x[1:], bins=[coarse, coarse])[0]
    J = J / J.sum()
    px, py = J.sum(1, keepdims=True), J.sum(0, keepdims=True)
    m = J > 0
    return float((J[m] * np.log(J[m] / (px @ py)[m])).sum())


def load_ettm1():
    """Real forecasting benchmark. 7 channels, quantised per-channel (each gets its
    own edges) then concatenated -- 6 splice points in ~480k tokens."""
    url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv"
    path = _download(url, "ETTm1.csv")
    raw = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=range(1, 8))
    tr_parts, va_parts = [], []
    for ch in range(raw.shape[1]):
        tr, va = quantize(raw[:, ch])
        tr_parts.append(tr); va_parts.append(va)
    return torch.cat(tr_parts), torch.cat(va_parts)


ENWIK8_PATH = "/content/titans-pytorch-src/data/enwik8.gz"

def load_enwik8(n_bytes=int(20e6)):
    """enwik8 ships inside the titans-pytorch repo (data/enwik8.gz, ~36 MB) -- the same
    source the existing real-text notebook uses. Shallow-clone it on first use, then hand
    off to the repo's own loader so byte handling and the train/val split are identical
    to the baseline runs."""
    if not os.path.exists(ENWIK8_PATH):
        print("  cloning titans-pytorch for data/enwik8.gz ...")
        os.system("git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git "
                  "/content/titans-pytorch-src")
    if not os.path.exists(ENWIK8_PATH):
        raise FileNotFoundError(
            f"{ENWIK8_PATH} not found. Clone manually:\n"
            "  !git clone --depth 1 https://github.com/lucidrains/titans-pytorch.git "
            "/content/titans-pytorch-src")
    return _load_enwik8_from_path(ENWIK8_PATH, n_bytes)


N_SYNTH = 400_000

def build_corpora(include_enwik8=True, include_ettm1=True):
    c = {}
    for phi in (0.0, 0.5, 0.9, 0.99):
        c[f"ar_phi{phi:.2f}"] = lambda phi=phi: quantize(ar1(N_SYNTH, phi))
    c["periodic"] = lambda: quantize(periodic(N_SYNTH))
    if include_ettm1:
        c["ETTm1"] = load_ettm1
    if include_enwik8:
        c["enwik8"] = load_enwik8
    return c

## 3b · Pre-flight: confirm the knob is real before spending GPU hours

Marginal entropy should be identical (~5.545 = log 256) for every corpus -- that is the
control working. Lag-1 MI should *vary*, and it is what the sweep manipulates. If these
two columns don't look like that, stop: the experiment has no independent variable.

In [ ]:
_pre = build_corpora(include_enwik8=False, include_ettm1=True)
print(f"{'corpus':<14}{'tokens':>10}{'marg.entropy':>14}{'lag-1 MI':>11}")
print(f"{'':<14}{'':>10}{'(want equal)':>14}{'(want varied)':>11}")
for _n, _l in _pre.items():
    _tr, _ = _l()
    print(f"{_n:<14}{len(_tr):>10,}{marginal_entropy_nats(_tr):>14.3f}{lag1_mi(_tr):>11.4f}")

## 4 · Model and training — imported from the repo

`build_model`, `sample_batch`, `train` and `_cos` come from
`experiments/titans_real_text.py` via the clone above. Nothing is redefined here, so a
change to that file changes this notebook too and the comparison against the enwik8
baseline stays valid by construction rather than by promise.

Only `final_val_loss` is local: the repo's `train` prints val loss as it goes but doesn't
return a final averaged number, which the sweep needs.

In [ ]:
@torch.no_grad()
def final_val_loss(model, data_val, seq_len, batch_size, device, n_batches=20):
    model.eval()
    return float(np.mean([model(sample_batch(data_val, seq_len, batch_size).to(device),
                                return_loss=True).item() for _ in range(n_batches)]))

## 5 · Measurement — the same memory probes, returning values instead of printing

`norm_ratio` is the end-of-document write magnitude over the early write magnitude, on
the units that actually moved. **Below 1 = forgetting, above 1 = accumulating.**

In [ ]:
# _cos and consecutive_write_alignment are imported from titans_real_text.
# measure_memory performs the SAME computation as that file's diff_memory_on_passage,
# but returns the numbers instead of printing them. The next cell verifies they agree.

@torch.no_grad()
def measure_memory(model, passage, device):
    """Early-vs-end weight-snapshot diff (same as diff_memory_on_passage) plus the
    consecutive-write alignment, returned as numbers."""
    model.eval()
    _, cache = model(passage.unsqueeze(0).to(device), return_cache=True)
    _, _, neural_mem_caches = cache
    state = neural_mem_caches[0]

    U0 = state.updates["model.weights.0"].detach()[0].cpu().numpy()
    U1 = state.updates["model.weights.1"].detach()[0].cpu().numpy()

    g_in, g_out = U0[1], U0[-1]
    gate_cos = _cos(g_in, g_out, axis=0)
    nr = (np.linalg.norm(g_out, axis=0) + 1e-9) / (np.linalg.norm(g_in, axis=0) + 1e-9)
    moved = gate_cos < 0.99

    consec = consecutive_write_alignment(model, passage, device)

    return {
        "norm_ratio":     float(nr[moved].mean()) if moved.any() else float("nan"),
        "units_moved":    int(moved.sum()),
        "gate_cos_median": float(np.median(gate_cos)),
        "write_align":    float(consec.mean()),
        "n_chunks":       int(U0.shape[0]),
    }


@torch.no_grad()
def measure_decay_gate(model, passage, device):
    """alpha_t on this corpus vs on random tokens, through the same trained weights."""
    mem_layer = next(group[4] for group in model.layers if group[4] is not None)
    captured = {}
    h = mem_layer.to_decay_factor.register_forward_hook(
        lambda m, i, o: captured.__setitem__("d", o.sigmoid().detach()))
    model(passage.unsqueeze(0).to(device), return_cache=True)
    real = captured["d"].mean().item()
    model(torch.randint(0, 256, passage.shape, device=device).unsqueeze(0), return_cache=True)
    rand = captured["d"].mean().item()
    h.remove()
    return {"gate_real": real, "gate_random": rand, "gate_gap": rand - real}

### Cross-check: does `measure_memory` agree with the repo's own function?

`diff_memory_on_passage` prints; `measure_memory` returns. They must compute the same
`norm_ratio`. Run this once on a throwaway model -- the printed "norm_ratio (end/early
write) on moved units" must match the returned value. If it doesn't, the sweep is
measuring something different from the baseline and the comparison is void.

In [ ]:
_m = build_model().to(DEVICE)
_tr, _va = quantize(ar1(20_000, 0.9))
_p = _va[:PASSAGE_LEN]

print("--- repo's diff_memory_on_passage (prints) ---")
diff_memory_on_passage(_m, _p, DEVICE)
print("\n--- measure_memory (returns) ---")
print(f"norm_ratio = {measure_memory(_m, _p, DEVICE)['norm_ratio']:.4f}")
print("\nThe two norm_ratio values above must match.")
del _m; torch.cuda.empty_cache()

## 6 · Run the sweep

In [ ]:
def run_corpus(name, loader, seed, checkpoints=CHECKPOINTS):
    """Train once, pausing at each checkpoint to measure. Cost is the longest
    checkpoint, not the sum -- training resumes where it left off."""
    print(f"\n=== {name}  (seed {seed}) ===")
    torch.manual_seed(seed); np.random.seed(seed)
    t0 = time.time()

    tr, va = loader()
    ent = marginal_entropy_nats(tr)
    mi  = lag1_mi(tr)
    print(f"  tokens: train {len(tr):,}  val {len(va):,}  | marginal entropy {ent:.3f} nats "
          f"(uniform = {np.log(256):.3f})  | lag-1 MI {mi:.4f}")

    model = build_model().to(DEVICE)
    passage = va[:PASSAGE_LEN]
    rows, done = [], 0

    for target in checkpoints:
        train(model, tr, va, target - done, SEQ_LEN, BATCH, LR, DEVICE)
        done = target
        row = {"corpus": name, "seed": seed, "steps": target,
               "val_loss": final_val_loss(model, va, SEQ_LEN, BATCH, DEVICE),
               "marginal_entropy": ent, "lag1_mi": mi,
               "minutes": (time.time() - t0) / 60}
        row.update(measure_memory(model, passage, DEVICE))
        row.update(measure_decay_gate(model, passage, DEVICE))
        print(f"  [{target:>5} steps]  val {row['val_loss']:.3f}  |  norm_ratio "
              f"{row['norm_ratio']:.2f}  |  moved {row['units_moved']}/{HIDDEN}"
              f"  |  {row['minutes']:.1f} min")
        rows.append(row)

    del model; torch.cuda.empty_cache()
    return rows


corpora = build_corpora(include_enwik8=True, include_ettm1=True)
results = []
if os.path.exists(RESULTS_PATH):
    results = json.load(open(RESULTS_PATH))
    print(f"resuming with {len(results)} existing rows")

done_pairs = {(r["corpus"], r["seed"]) for r in results}
for seed in SEEDS:
    for name, loader in corpora.items():
        if (name, seed) in done_pairs:
            print(f"skip {name} seed {seed} (already done)"); continue
        results.extend(run_corpus(name, loader, seed))
        json.dump(results, open(RESULTS_PATH, "w"), indent=2)   # checkpoint every corpus

print(f"\ndone — {len(results)} rows")

## 7 · The table

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
HEADLINE_STEPS = df.steps.max()
print(f"headline table at {HEADLINE_STEPS} steps "
      f"(checkpoints available: {sorted(df.steps.unique())})")

summary = (df[df.steps == HEADLINE_STEPS].groupby("corpus")
             .agg(lag1_mi=("lag1_mi", "mean"),
                  val_loss=("val_loss", "mean"),
                  norm_ratio=("norm_ratio", "mean"),
                  norm_ratio_sd=("norm_ratio", "std"),
                  units_moved=("units_moved", "mean"),
                  write_align=("write_align", "mean"),
                  gate_gap=("gate_gap", "mean"),
                  n=("seed", "count"))
             .sort_values("val_loss"))
summary.round(3)

## 8 · The plot

One question, one axis: **does `norm_ratio` track predictability?** x is val loss (low =
predictable), y is `norm_ratio` with the forget/accumulate boundary drawn at 1.0.
The AR(1) family is one hue stepped by phi, since phi is an ordered magnitude, not an
identity. Every point is directly labelled — identity is never colour-alone.

In [ ]:
# AR(1) family: one hue, stepped by phi (ordinal ramp, no lighter than step 250)
AR_RAMP = {"ar_phi0.00": "#86b6ef", "ar_phi0.50": "#5598e7",
           "ar_phi0.90": "#2a78d6", "ar_phi0.99": "#184f95"}
# distinct categorical hues for the non-AR corpora (validated, all-pairs)
OTHER   = {"periodic": "#eb6834", "ETTm1": "#1baf7a", "enwik8": "#4a3aa7"}
COLORS  = {**AR_RAMP, **OTHER}

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")

ax.axhline(1.0, color="#9a9a93", lw=1, ls="--", zorder=1)
ax.text(ax.get_xlim()[0], 1.0, "  accumulates above / forgets below",
        va="bottom", ha="left", fontsize=9, color="#6b6b64")

for corpus, row in summary.iterrows():
    ax.scatter(row.val_loss, row.norm_ratio, s=110,
               color=COLORS.get(corpus, "#6b6b64"),
               edgecolor="#fcfcfb", linewidth=2, zorder=3)
    ax.annotate(corpus, (row.val_loss, row.norm_ratio),
                textcoords="offset points", xytext=(9, 0), va="center",
                fontsize=9.5, color="#33332f")

ax.set_xlabel("validation loss (nats) — lower = more predictable", fontsize=10, color="#54544c")
ax.set_ylabel("norm_ratio (end / early write)", fontsize=10, color="#54544c")
ax.set_title("Does memory accumulation track how predictable the data is?",
             fontsize=12, color="#33332f", pad=14, loc="left")
ax.grid(True, color="#e8e8e3", lw=0.8); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color("#d4d4cd")
ax.tick_params(colors="#6b6b64", labelsize=9)

from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color="#2a78d6",
                          label="AR(1), shade = phi"),
                   Line2D([], [], marker="o", ls="", color="#eb6834", label="periodic"),
                   Line2D([], [], marker="o", ls="", color="#1baf7a", label="ETTm1 (real)"),
                   Line2D([], [], marker="o", ls="", color="#4a3aa7", label="enwik8 (text)")],
          frameon=False, fontsize=9, labelcolor="#54544c", loc="best")
plt.tight_layout(); plt.savefig("norm_ratio_vs_predictability.png", dpi=200); plt.show()

In [ ]:
# Companion view: norm_ratio by corpus, ordered — the same numbers as magnitude
order = summary.sort_values("norm_ratio")
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")

bars = ax.barh(range(len(order)), order.norm_ratio, height=0.62,
               color=[COLORS.get(c, "#6b6b64") for c in order.index])
ax.axvline(1.0, color="#9a9a93", lw=1, ls="--")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order.index, fontsize=9.5, color="#33332f")
for i, v in enumerate(order.norm_ratio):
    ax.text(v + 0.03, i, f"{v:.2f}", va="center", fontsize=9, color="#54544c")

ax.set_xlabel("norm_ratio (end / early write)", fontsize=10, color="#54544c")
ax.set_title("Forgetting vs accumulation by corpus", fontsize=12, color="#33332f", pad=12, loc="left")
ax.grid(True, axis="x", color="#e8e8e3", lw=0.8); ax.set_axisbelow(True)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#d4d4cd")
ax.tick_params(colors="#6b6b64", labelsize=9)
plt.tight_layout(); plt.savefig("norm_ratio_by_corpus.png", dpi=200); plt.show()

### Is the ordering stable across training, or an artifact of where we stopped?

`norm_ratio` grows with training (the enwik8 baseline went 1.42 at 1200 steps to 2.20 at
3000). Predictable corpora also converge sooner. So a single fixed cutoff could confuse
"more predictable" with "further along". If the corpora keep the same *order* at every
checkpoint, the relationship is not an artifact of the stopping point.

In [ ]:
traj = df.groupby(["corpus", "steps"]).norm_ratio.mean().unstack()
display(traj.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")
ax.axhline(1.0, color="#9a9a93", lw=1, ls="--")
for corpus in traj.index:
    ax.plot(traj.columns, traj.loc[corpus], marker="o", ms=6, lw=2,
            color=COLORS.get(corpus, "#6b6b64"))
    ax.annotate(corpus, (traj.columns[-1], traj.loc[corpus].iloc[-1]),
                textcoords="offset points", xytext=(8, 0), va="center",
                fontsize=9, color="#33332f")
ax.set_xlabel("training steps", fontsize=10, color="#54544c")
ax.set_ylabel("norm_ratio", fontsize=10, color="#54544c")
ax.set_title("Does the ordering hold at every checkpoint?", fontsize=12,
             color="#33332f", pad=12, loc="left")
ax.grid(True, color="#e8e8e3", lw=0.8); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color("#d4d4cd")
ax.tick_params(colors="#6b6b64", labelsize=9)
ax.set_xlim(right=traj.columns[-1] * 1.22)
plt.tight_layout(); plt.savefig("norm_ratio_trajectory.png", dpi=200); plt.show()

rank_by_step = {s: list(traj[s].sort_values().index) for s in traj.columns}
stable = len({tuple(v) for v in rank_by_step.values()}) == 1
print("\nordering identical at every checkpoint:", stable)
if not stable:
    for s, order in rank_by_step.items(): print(f"  {s:>5}: {order}")

## 9 · How to read the result

Check the correlation between `val_loss` and `norm_ratio` across corpora:

```python
print(summary[["val_loss", "norm_ratio"]].corr(method="spearman"))
```

**If `norm_ratio` rises as val loss falls** — predictable data accumulates, unpredictable
data forgets — then the §6 open question has an answer: the gate's learned behaviour is
set by how predictable its training corpus was, which is exactly the "baked in by what
the memory was trained on" hypothesis, now with a controlled sweep behind it. The AR(1)
family alone carries this, since phi is the only thing that changes across those four.

**If `norm_ratio` is flat across the AR sweep but text and ETTm1 still differ**, then
predictability is not the driver and something structural about those corpora is —
narrower, still a result, and it rules out the most obvious remaining explanation.

**If enwik8 lands near 2.2**, the pipeline reproduces the existing finding and the new
points are trustworthy. Check this first — it is the control.

### What each corpus can and cannot support

- **AR(1) x4** -- marginals matched, only phi moves. A monotone trend here is a clean
  causal claim about temporal predictability. This is the result.
- **periodic, ETTm1** -- marginals matched to the AR group, so they extend the claim,
  but each differs from AR in more than one way. Supporting, not decisive.
- **enwik8** -- marginal NOT matched (raw bytes). Use it only to confirm the pipeline
  reproduces ~2.2. A text-vs-AR gap is confounded and cannot be attributed to
  temporal structure.

Secondary diagnostics: `units_moved` and `gate_gap`. A corpus where almost no units
move makes its `norm_ratio` less meaningful, so check that column before trusting an
outlier.